In [1]:
import sys
import os

In [2]:
notebook_dir = os.getcwd()
src_dir = notebook_dir  # The notebook is already in src/
sys.path.insert(0, src_dir)

print(f"Added to path: {src_dir}")
print(f"Current working directory: {notebook_dir}")

Added to path: e:\Documents\Programming\Projects\Reinforcement\PhoenX_RL\src
Current working directory: e:\Documents\Programming\Projects\Reinforcement\PhoenX_RL\src


In [ ]:
import yaml

In [ ]:
with open('E:\Documents\Programming\Projects\Reinforcement\PhoenX_RL\src\Configs\ddpg.yml', 'r') as f:
    config = yaml.safe_load(f)

In [ ]:
config

In [ ]:
config['models']['actor']['layer_config'][0]

In [3]:
import gymnasium as gym
import app.agent_utils

e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment Reacher-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment Pusher-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment InvertedPendulum-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment InvertedDoublePendulum-v2 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
e:\Miniconda3\envs\rl_env\Lib\site-packages\gymnasium\env

In [ ]:
import numpy as np

In [ ]:
states, infos = env_wrap.env.env.reset()

In [ ]:
next_states, rewards, terms, truncs, infos = env_wrap.env.env.step(env_wrap.env.env.action_space.sample())

In [ ]:
env_wrap.env.env.step(env_wrap.env.env.action_space.sample())

In [ ]:
vec_env = gym.make_vec(
            id="FrozenLake-v1",
            num_envs=4,
            vectorization_mode="sync",
            vector_kwargs={"autoreset_mode": "SameStep"},
        )

In [6]:
env = app.agent_utils.GymnasiumWrapper(
    cfg="Taxi-v3",
    num_envs=4,
    wrappers=[{"type": "OneHotObservationWrapper", "params": {}}],
    render_mode=None,
    seed=42,
)

In [8]:
env.single_observation_space.shape[0]

500

In [ ]:
states

In [ ]:
states_buffer = []
actions_buffer = []
for i in range(100):
    action = env.action_space.sample()
    next_state, reward, done, info = env.step(action)
    states_buffer.append(next_state)
    actions_buffer.append(action)


In [ ]:
states_buffer

In [ ]:
actions_buffer

In [ ]:
from app.env_wrapper import OneHotObservationWrapper
wrapped_env = OneHotObservationWrapper(vec_env)

In [ ]:
vec_env.single_observation_space.n

In [ ]:
vec_env.single_observation_space.sample()

In [ ]:
space = vec_env.single_observation_space
int(np.prod(space.shape))

In [ ]:
env_wrap = app.agent_utils.GymnasiumWrapper(
    cfg="LunarLanderContinuous-v3",
    num_envs=4,
    wrappers=[{"type": "VectorNStepReward", "params": {"n": 3}}],
    render_mode=None,
    seed=42,
    obs_key=None,
    goal_key=None
)

In [ ]:
states, infos = env_wrap.reset()

In [ ]:
infos

In [ ]:
steps = 0
while True:
    steps += 1
    states, rewards, dones, infos = env_wrap.step(env_wrap.action_space.sample())
    if any(dones):
        break


In [ ]:
infos

In [ ]:
states, rewards, dones, infos = env_wrap.step(env_wrap.action_space.sample())

In [ ]:
infos

In [ ]:
dones

In [ ]:
states

In [ ]:
rewards

In [ ]:
infos

In [ ]:
infos['n-step trajectory']['next_states'].shape

In [ ]:
infos.get("_final_obs", [False]*4)[1]

In [ ]:
infos['final_obs'][1]

In [ ]:
states

In [ ]:
from app.rl_agents import load_agent

In [ ]:
ddpg_agent = load_agent("E:\Documents\Programming\Projects\Reinforcement\PhoenX_RL\src\Trained_Models\LunarLanderContinuous-v3_N3_E4_2\DDPG", False)

In [ ]:
ddpg_agent.callbacks[0].run_name = 'train-49'

In [ ]:
ddpg_agent.train(1000, steps_per_learn=1, render_freq=100, seed=42)

In [ ]:
ddpg_agent.replay_buffer.sample(100)

In [ ]:
states, actions, rewards, next_states, dones, traj_lengths = ddpg_agent.replay_buffer.sample(10240)

In [ ]:
import numpy as np
np.savez(
    "buffer_sample.npz",
    states=states.cpu().numpy(),
    actions=actions.cpu().numpy(),
    rewards=rewards.cpu().numpy(),
    next_states=next_states.cpu().numpy(),
    dones=dones.cpu().numpy(),
    traj_lengths=traj_lengths.cpu().numpy()
)

In [ ]:
rewards.shape

In [ ]:
n_states = ddpg_agent.env.env.n_states

In [ ]:
n_states

In [ ]:
formatted_states = ddpg_agent.env.env.format_trajectory(n_states, pad_mode="repeat")

In [ ]:
n_rewards = ddpg_agent.env.env.n_rewards

In [ ]:
n_rewards

In [ ]:
import torch as T
def format_trajectory(trajectory, pad_mode:str|T.Tensor="repeat"):
        """Format trajectory from per-env deques to batched tensor.
        
        Args:
            trajectory: List of deques containing tensors.
            pad_mode: Mode to pad the trajectory. "repeat" to repeat the last value or tensor to pad with passed value.

        Returns:
            Tensor: Batched trajectory.
        """
        trajs = []
        for d in trajectory:
            seq = list(d)
            print(seq)
            if pad_mode == "repeat":
                padding = seq[-1]
            else:
                padding = pad_mode
            while len(seq) < 3:
                seq.append(padding)
            trajs.append(T.stack(seq, dim=0))
        return T.stack(trajs, dim=0)

In [ ]:
format_trajectory(n_states, pad_mode="repeat")

In [ ]:
n_dones = ddpg_agent.env.env.n_dones
n_dones

In [ ]:
formatted_dones = ddpg_agent.env.env.format_trajectory(n_dones, ddpg_agent.env.env._pad_reward)

In [ ]:
formatted_dones

In [ ]:
n_rewards = ddpg_agent.env.env.n_rewards
formatted_rewards = ddpg_agent.env.env.format_trajectory(n_rewards, ddpg_agent.env.env._pad_reward)

In [ ]:
formatted_rewards

In [ ]:
states

In [ ]:
states[42]

In [ ]:
next_states[42]

In [ ]:
rewards[42]

In [ ]:
dones[42]

In [ ]:
(dones.sum(dim=1) == 0).float()

In [ ]:
idx = (dones[:, -1] == 1).nonzero(as_tuple=True)

In [ ]:
idx

In [ ]:
compute_n_step_return(rewards, 0.99, device="cpu")

In [ ]:
from app.agent_utils import compute_n_step_return

In [ ]:
returns = compute_n_step_return(rewards, 0.99, device="cpu")

In [ ]:
returns[0]

In [ ]:
rewards = [
    deque(iterable=[T.tensor(0), T.tensor(1)], maxlen=3),
    deque(iterable=[T.tensor(0), T.tensor(0), T.tensor(1)], maxlen=3),
    deque(iterable=[T.tensor(0), T.tensor(0)], maxlen=3),
    deque(iterable=[T.tensor(1)], maxlen=3),
]

trajectory = _format_trajectory(rewards, pad_with_ones=False)